In [27]:
from pathlib import Path
 
import numpy
from numpy.polynomial import Polynomial
import pandas

from pvfit.common import E_hemispherical_tilted_W_per_m2_stc, T_degC_stc
from pvfit.measurement.iv.types import IVPerformanceMatrix
from pvfit.modeling.dc.common import Material
import pvfit.modeling.dc.single_diode.model.simple.inference_matrix as sdm_simple_inf_matrix
import pvfit.modeling.dc.single_diode.model.photoconductive_shunt.inference_matrix as sdm_ps_inf_matrix

In [21]:
path = Path(
    "src/pvfit/modeling/dc/single_diode/model/demos/data/"
    "Sandia_PV_Module_P-Matrix-and-TempCo-Data_2019.xlsx"
)
sheets_matrix = pandas.read_excel(
    path, sheet_name=None, skiprows=tuple(range(4)), header=0, skipfooter=16
)
sheets_coeffs = pandas.read_excel(
    path, sheet_name=None, skiprows=tuple(range(34)), header=0
)
N_s = {
    "19074-001": 96,
    "19074-002": 60,
    "19074-003": 60,
    "19074-004": 60,
    "19074-005": 60,
    "19074-006": 60,
    "19074-007": 60,
    "19074-008": 60,
    "19074-009": 72,
}
cell_tech = {
    "19074-001": "HIT Mono",
    "19074-002": "N-PERT Si",
    "19074-003": "poly-Si PERC",
    "19074-004": "poly-Si",
    "19074-005": "mono-Si",
    "19074-006": "poly-Si",
    "19074-007": "mono-Si PERC",
    "19074-008": "mono-Si PERC",
    "19074-009": "mono-Si PERC",
}

In [30]:
print(sheets_coeffs["19074-001 (VBHN325SA 16)"].columns)
T_degC_data = sheets_coeffs["19074-001 (VBHN325SA 16)"].iloc[1:4, 1]
I_sc_A_data = sheets_coeffs["19074-001 (VBHN325SA 16)"].iloc[1:4, 3]
P_mp_W_data = sheets_coeffs["19074-001 (VBHN325SA 16)"].iloc[1:4, 7]
V_oc_V_data = sheets_coeffs["19074-001 (VBHN325SA 16)"].iloc[1:4, 4]

dI_sc_dT_A_per_degC_0 = Polynomial.fit(T_degC_data, I_sc_A_data, deg=1).convert().coef[1]
dP_mp_dT_W_per_degC_0 = Polynomial.fit(T_degC_data, P_mp_W_data, deg=1).convert().coef[1]
dV_oc_dT_V_per_degC_0 = Polynomial.fit(T_degC_data, V_oc_V_data, deg=1).convert().coef[1]

dI_sc_dT_A_per_degC_0, dP_mp_dT_W_per_degC_0, dV_oc_dT_V_per_degC_0

Index(['Module ID', 'Measured Temperature [Â°C]', 'Irradiance [W/m2]',
       'Isc [A]', 'Voc [V]', 'Imp [A]', 'Vmp [V]', 'Pmp [W]'],
      dtype='str')


(np.float64(0.0019897729336658195),
 np.float64(-0.8588767550721548),
 np.float64(-0.16719963536730478))

In [ ]:
for sheet_key, sheet_value in sheets_matrix.items():
    print(sheet_key)

    iv_performance_matrix = IVPerformanceMatrix(
        material=Material.monoSi,
        N_s=N_s[sheet_key.split(" ")[0]],
        I_sc_A=sheet_value.iloc[:, 3].to_numpy(),
        I_mp_A=sheet_value.iloc[:, 5].to_numpy(),
        V_mp_V=sheet_value.iloc[:, 6].to_numpy(),
        V_oc_V=sheet_value.iloc[:, 4].to_numpy(),
        E_W_per_m2=sheet_value.iloc[:, 2].to_numpy(),
        T_degC=sheet_value.iloc[:, 1].to_numpy(),
        E_W_per_m2_0=E_hemispherical_tilted_W_per_m2_stc,
        T_degC_0=T_degC_stc,
    )

    model_parameters_matrix_simple = sdm_simple_inf_matrix.fit(
        iv_performance_matrix=iv_performance_matrix,
    ).model_parameters
    print(model_parameters_matrix_simple)
    print(
        "compute_fit_quality:",
        sdm_simple_inf_matrix.compute_fit_quality(
            iv_performance_matrix=iv_performance_matrix,
            model_parameters=model_parameters_matrix_simple,
        )[0]["mape"]
    )

    model_parameters_matrix_ps = sdm_ps_inf_matrix.fit(
        iv_performance_matrix=iv_performance_matrix,
    ).model_parameters
    print(model_parameters_matrix_ps)
    print(
        "compute_fit_quality:",
        sdm_ps_inf_matrix.compute_fit_quality(
            iv_performance_matrix=iv_performance_matrix,
            model_parameters=model_parameters_matrix_simple,
        )[0]["mape"]
    )
    print("")

19074-001 (VBHN325SA 16)
ModelParameters(N_s=96, T_degC_0=25.0, I_sc_A_0=5.90284948722764, I_rs_A_0=2.610882528587242e-11, n_0=1.0919174227594546, R_s_Ohm_0=0.6225067581121211, G_p_S_0=0.004805876850654534, E_g_eV_0=1.1633500315882699)
compute_fit_quality: {'I_sc_A': np.float64(1.2335811384723961e-15), 'I_mp_A': np.float64(10.446411212652047), 'P_mp_W': np.float64(11.027273516875526), 'V_mp_V': np.float64(0.9108570239687713), 'V_oc_V': np.float64(0.7228436865087319)}
ModelParameters(N_s=96, T_degC_0=25.0, I_sc_A_0=5.90284948722764, I_rs_A_0=4.2395761223699417e-10, n_0=1.2230977387213517, R_s_Ohm_0=0.48879469355774774, G_p_S_0=0.003363388865384774, E_g_eV_0=1.1522582036231217)
compute_fit_quality: {'I_sc_A': np.float64(1.2335811384723961e-15), 'I_mp_A': np.float64(1.6151407997222165), 'P_mp_W': np.float64(1.9062739273409883), 'V_mp_V': np.float64(0.7874286696490925), 'V_oc_V': np.float64(0.15323091223688529)}

19074-002 (LG320N1K-A5)
ModelParameters(N_s=60, T_degC_0=25.0, I_sc_A_0=10.35

In [ ]:
# Task 13 module.
iv_performance_matrix = IVPerformanceMatrix(
    material=Material.monoSi,
    N_s=96,  # Inferred from Voc.
    I_sc_A=numpy.array(
        (
            0.94, 1.89, 3.78, 5.67, 7.56, 9.44,
            0.95, 1.90, 3.79, 5.69, 7.59, 9.49, 10.43,
                        3.84, 5.75, 7.67, 9.60, 10.55,
                              5.82, 7.76, 9.71, 10.66,
        )
    ),
    I_mp_A=numpy.array(),  # TODO Convert from P_mpW.
    V_mp_V=numpy.array(),  # TODO Convert from P_mpW.
    V_oc_V=numpy.array(
        (
            36.72, 37.92, 39.16, 39.90, 40.43, 40.85,
            35.45, 36.70, 37.98, 38.74, 39.29, 39.71, 39.90,
                          35.01, 35.84, 36.42, 36.89, 37.08,
                                 32.93, 33.56, 34.05, 34.26,	
        )
    ),
    E_W_per_m2=numpy.array(
        (
            100, 200, 400, 600, 800, 1000,
            100, 200, 400, 600, 800, 1000, 1100,
                      400, 600, 800, 1000, 1100,
                           600, 800, 1000, 1100,
        )
    ),
    T_degC=numpy.array(
        (
            15, 15, 15, 15, 15, 15,
            25, 25, 25, 25, 25, 25, 25,
                    50, 50, 50, 50, 50,
                        75, 75, 75, 75,
        )
    ),
    E_W_per_m2_0=E_hemispherical_tilted_W_per_m2_stc,
    T_degC_0=T_degC_stc,
)